# Inspect Tip Feature Training Dataset

Read the Task 2 Delta output and confirm it is ready for model training.

In [1]:
from pyspark.ml.functions import vector_to_array
from pyspark.sql import SparkSession, functions as F

FEATURE_TABLE_PATH = "s3a://delta-data/predictions/feature_training"

spark = (
    SparkSession.builder
    .master("spark://spark-master:7077")
    .appName("InspectTipFeatureTraining")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
spark.conf.set("spark.sql.session.timeZone", "UTC")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 11:26:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
feature_training = spark.read.format("delta").load(FEATURE_TABLE_PATH)

expected_columns = {"features", "tip_amount", "dataset_split"}
actual_columns = set(feature_training.columns)

print(f"Row count: {feature_training.count():,}")
print(f"Columns: {feature_training.columns}")
print(f"Expected columns only: {actual_columns == expected_columns}")
feature_training.printSchema()

26/10/03 11:26:58 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.
26/10/03 11:26:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Row count: 7,734,323
Columns: ['features', 'tip_amount', 'dataset_split']
Expected columns only: True
root
 |-- features: vector (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- dataset_split: string (nullable = true)



In [3]:
split_counts = (
    feature_training
    .groupBy("dataset_split")
    .count()
    .orderBy("dataset_split")
)
split_counts.show()

expected_splits = {"train", "validation", "test"}
observed_splits = {row["dataset_split"] for row in split_counts.collect()}
print(f"Observed splits: {sorted(observed_splits)}")
print(f"Only expected split values: {observed_splits <= expected_splits}")

+-------------+-------+
|dataset_split|  count|
+-------------+-------+
|         test|1161730|
|        train|5411459|
|   validation|1161134|
+-------------+-------+

Observed splits: ['test', 'train', 'validation']
Only expected split values: True


In [4]:
(
    feature_training
    .groupBy("dataset_split")
    .agg(
        F.count("*").alias("rows"),
        F.sum(F.col("tip_amount").isNull().cast("int")).alias("null_tip_amounts"),
        F.sum(F.col("features").isNull().cast("int")).alias("null_feature_vectors"),
        F.min("tip_amount").alias("min_tip_amount"),
        F.avg("tip_amount").alias("average_tip_amount"),
        F.max("tip_amount").alias("max_tip_amount"),
    )
    .orderBy("dataset_split")
    .show(truncate=False)
)

+-------------+-------+----------------+--------------------+--------------+------------------+--------------+
|dataset_split|rows   |null_tip_amounts|null_feature_vectors|min_tip_amount|average_tip_amount|max_tip_amount|
+-------------+-------+----------------+--------------------+--------------+------------------+--------------+
|test         |1161730|0               |0                   |0.0           |4.207904934881103 |255.0         |
|train        |5411459|0               |0                   |0.0           |4.210745678753918 |999.99        |
|validation   |1161134|0               |0                   |0.0           |4.202199746109761 |428.0         |
+-------------+-------+----------------+--------------------+--------------+------------------+--------------+



In [5]:
feature_dimensions = feature_training.select(
    F.size(vector_to_array("features")).alias("feature_dimension")
)

feature_dimensions.groupBy("feature_dimension").count().orderBy("feature_dimension").show()

+-----------------+-------+
|feature_dimension|  count|
+-----------------+-------+
|               83|7734323|
+-----------------+-------+



In [6]:
(
    feature_training
    .select(
        "dataset_split",
        "tip_amount",
        F.size(vector_to_array("features")).alias("feature_dimension"),
        "features",
    )
    .orderBy("dataset_split")
    .show(10, truncate=False)
)

+-------------+----------+-----------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|dataset_split|tip_amount|feature

In [7]:
spark.stop()